# Dirac Notation, Decoded

Dirac notation looks intimidating. It is not. It is compact NumPy with stylish brackets. By the end of this notebook you will translate any Dirac expression into code on sight.

**Objectives:**
- Build a one-to-one mental map: Dirac symbol ↔ NumPy expression
- Compute inner products, outer products, and tensor products in both notations
- Read expressions like $\langle 0 | H | + \rangle$ without slowing down

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

# Our reference vocabulary.
zero  = np.array([1, 0], dtype=complex)
one   = np.array([0, 1], dtype=complex)
plus  = (zero + one)  / np.sqrt(2)
minus = (zero - one)  / np.sqrt(2)
H = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
I = np.eye(2, dtype=complex)

## 1. Kets and bras

- A **ket** $|\psi\rangle$ is a column vector. In NumPy, a 1-D array works fine.
- A **bra** $\langle \psi |$ is its conjugate transpose — a row vector with   conjugated entries.

Why two names for what is essentially the same data? Because pairing a bra with a ket gives an inner product, which is a number, while pairing a ket with a bra gives an outer product, which is a matrix. The notation makes which is which obvious.

In [ ]:
psi = (1/np.sqrt(2)) * np.array([1, 1j])
print('ket |psi>:', psi)
print('bra <psi|:', psi.conj())

## 2. Inner product $\langle a | b \rangle$

This is `a.conj() @ b`. Returns a single complex number that measures overlap.

In [ ]:
print('<0|0> =', zero.conj() @ zero)        # 1
print('<0|1> =', zero.conj() @ one)         # 0
print('<0|+> =', zero.conj() @ plus)        # 1/sqrt(2)
print('<+|-> =', plus.conj() @ minus)       # 0  — plus and minus are orthogonal

## 3. Applying a gate: $U|\psi\rangle$

This is just matrix-vector multiplication: `U @ psi`.

In [ ]:
print('X|0> =', X @ zero)          # |1>
print('X|1> =', X @ one)           # |0>
print('H|0> =', H @ zero)          # |+>
print('H|+> =', H @ plus)          # |0>  — H is its own inverse

## 4. Sandwiches: $\langle a | U | b \rangle$

Read right-to-left: first apply `U` to `|b>`, then take the inner product with `<a|`. In code: `a.conj() @ (U @ b)` — but `@` is associative, so `a.conj() @ U @ b` works too.

In [ ]:
# Examples
print('<0|H|0> =', zero.conj() @ H @ zero)     # 1/sqrt(2)
print('<1|H|0> =', one.conj()  @ H @ zero)     # 1/sqrt(2)
print('<0|H|+> =', zero.conj() @ H @ plus)     # 1
print('<0|X|0> =', zero.conj() @ X @ zero)     # 0

## 5. Outer products $|a\rangle\langle b|$

These are matrices, not numbers. In NumPy: `np.outer(a, b.conj())`. The **projector onto $|\psi\rangle$** is $|\psi\rangle\langle\psi|$ — used constantly in measurement theory.

In [ ]:
P0 = np.outer(zero, zero.conj())          # |0><0|
print('projector onto |0>:')
print(P0)

# Acting on a state: P0 |+> = |0><0|+> = (1/sqrt(2)) |0>
print('\nP0 |+> =', P0 @ plus)

## 6. Tensor products $|a\rangle \otimes |b\rangle$

Same as notebook 1: `np.kron(a, b)`. Often written without the `⊗` symbol — for example, `|01>` is shorthand for `|0> ⊗ |1>`.

In [ ]:
# |01> in conventional notation, equivalent to np.kron(zero, one)
psi_01 = np.kron(zero, one)
print('|01> =', psi_01)

# A simple entangled state: (|00> + |11>) / sqrt(2)  — the Bell pair
bell = (np.kron(zero, zero) + np.kron(one, one)) / np.sqrt(2)
print('Bell state =', bell)
print('measurement probs:', np.abs(bell)**2)   # 50% |00>, 50% |11>, never |01> or |10>

## 7. The full Rosetta stone

Pin this in your head:

| Dirac | NumPy | Type |
|---|---|---|
| $|0\rangle, |1\rangle$ | `np.array([1,0])`, `np.array([0,1])` | ket (vector) |
| $\langle \psi |$ | `psi.conj()` | bra (row vector) |
| $\langle a | b \rangle$ | `a.conj() @ b` | scalar |
| $\| \psi \|^2 = \langle \psi | \psi \rangle$ | `psi.conj() @ psi` | scalar (real, $\geq 0$) |
| $U|\psi\rangle$ | `U @ psi` | ket |
| $\langle a | U | b \rangle$ | `a.conj() @ U @ b` | scalar |
| $\| a \rangle \langle b \|$ | `np.outer(a, b.conj())` | matrix |
| $\| a \rangle \otimes \| b \rangle$ | `np.kron(a, b)` | longer ket |
| $A \otimes B$ | `np.kron(A, B)` | bigger matrix |
| $U^\dagger$ | `U.conj().T` | matrix |
| Born rule: P(outcome k) | `abs(psi[k])**2` | probability |


## 8. Self-check

In [ ]:
# Q1: Compute <+|H|0>. What number do you expect?
# TODO


In [ ]:
# Q2: Build the projector onto |+> as a matrix. Verify P @ |+> == |+>
# and P @ |-> == 0.
# TODO


In [ ]:
# Q3: Build |+> ⊗ |0> and compute its measurement probabilities (as a 4-vector).
# Which outcomes have nonzero probability and what are they?
# TODO


### Solutions

In [ ]:
# --- Q1 ---
print(plus.conj() @ H @ zero)     # 1  (because H|0> = |+>, and <+|+> = 1)

# --- Q2 ---
P_plus = np.outer(plus, plus.conj())
print('P|+> =', P_plus @ plus)    # |+>
print('P|-> =', P_plus @ minus)   # zero vector

# --- Q3 ---
state = np.kron(plus, zero)
probs = np.abs(state) ** 2
print('probs over (|00>, |01>, |10>, |11>):', probs)
# 50% |00>, 50% |10> — the second qubit is always 0, the first is 50/50

**You finished notebook 5.** Move on to [`06-bloch-sphere-playground.ipynb`](06-bloch-sphere-playground.ipynb).